# 🦀 Day 6: Core Logic — Part 1: In-Memory Storage Engine
---

Today we implement the in-memory storage engine for RustKV.

- **MemoryStorage**: Implements `trait Storage` with HashMap backend
- **GET**: Lookup + check expiration (lazy)
- **SET**: Insert with optional TTL
- **DELETE**: Remove entry
- **KEYS**: Pattern matching with glob-style wildcards
- **Thread-safe**: `Arc<RwLock<HashMap>>` for concurrent access
- **Expiration**: Lazy (on access) vs active (periodic cleanup)

In [ ]:
use std::collections::HashMap;
use std::sync::{Arc, RwLock};
use std::time::{Duration, Instant};

// Core types (from Day 5)
#[derive(Debug, Clone, PartialEq)]
pub enum Value {
    String(String),
    Integer(i64),
}

pub struct Entry {
    pub value: Value,
    pub created_at: Instant,
    pub expires_at: Option<Instant>,
}

impl Entry {
    fn is_expired(&self) -> bool {
        self.expires_at.map_or(false, |t| Instant::now() >= t)
    }
}

pub trait Storage: Send + Sync {
    fn get(&self, key: &str) -> Option<Value>;
    fn set(&self, key: &str, value: Value, ttl: Option<Duration>);
    fn delete(&self, key: &str) -> bool;
    fn keys(&self, pattern: &str) -> Vec<String>;
}

In [ ]:
/// In-memory storage backed by HashMap
pub struct MemoryStorage {
    data: Arc<RwLock<HashMap<String, Entry>>>,
}

impl MemoryStorage {
    pub fn new() -> Self {
        Self {
            data: Arc::new(RwLock::new(HashMap::new())),
        }
    }
}

impl Default for MemoryStorage {
    fn default() -> Self {
        Self::new()
    }
}

impl Storage for MemoryStorage {
    fn get(&self, key: &str) -> Option<Value> {
        let guard = self.data.read().unwrap();
        guard.get(key).and_then(|e| {
            if e.is_expired() {
                None
            } else {
                Some(e.value.clone())
            }
        })
    }

    fn set(&self, key: &str, value: Value, ttl: Option<Duration>) {
        let created = Instant::now();
        let expires_at = ttl.map(|d| created + d);
        let entry = Entry {
            value,
            created_at: created,
            expires_at,
        };
        let mut guard = self.data.write().unwrap();
        guard.insert(key.to_string(), entry);
    }

    fn delete(&self, key: &str) -> bool {
        let mut guard = self.data.write().unwrap();
        guard.remove(key).is_some()
    }

    fn keys(&self, pattern: &str) -> Vec<String> {
        let guard = self.data.read().unwrap();
        if pattern == "*" {
            guard.keys().cloned().collect()
        } else {
            // Simple prefix match (full glob in exercise)
            let prefix = pattern.trim_end_matches('*');
            guard.keys().filter(|k| k.starts_with(prefix)).cloned().collect()
        }
    }
}

println!("MemoryStorage implemented!");

In [ ]:
// Tests for MemoryStorage

let store = MemoryStorage::new();

// SET and GET
store.set("name", Value::String("RustKV".into()), None);
assert_eq!(store.get("name"), Some(Value::String("RustKV".into())));

// SET with TTL
store.set("temp", Value::Integer(42), Some(Duration::from_secs(1)));
assert_eq!(store.get("temp"), Some(Value::Integer(42)));

// DELETE
store.delete("name");
assert_eq!(store.get("name"), None);

// KEYS
store.set("user:1", Value::String("alice".into()), None);
store.set("user:2", Value::String("bob".into()), None);
store.set("config", Value::String("x".into()), None);
let keys = store.keys("user:*");
assert!(keys.contains(&"user:1".to_string()));
assert!(keys.contains(&"user:2".to_string()));
assert!(!keys.contains(&"config".to_string()));

println!("All MemoryStorage tests passed!");

## Lazy vs Active Expiration

| Strategy | When | Pros | Cons |
|----------|------|------|------|
| **Lazy** | Check on GET | Simple, no background task | Expired keys stay until accessed |
| **Active** | Periodic cleanup | Memory reclaimed sooner | Extra thread, more complexity |

RustKV uses **lazy** expiration for simplicity. Optionally add a background task later.

## 📝 Exercise: Implement KEYS with Full Glob Pattern Matching

Extend `keys()` to support:
- `*` — match any characters
- `?` — match single character
- `user:*` — prefix match (already done)
- `*:42` — suffix match

Hint: Convert glob pattern to regex or implement a simple matcher.

In [ ]:
// YOUR CODE HERE
// Implement glob-style pattern matching for keys()

fn simple_glob_match(pattern: &str, key: &str) -> bool {
    // Trivial implementation: * matches anything
    if pattern == "*" {
        return true;
    }
    // Split by * and check segments
    let parts: Vec<&str> = pattern.split('*').collect();
    if parts.len() == 1 {
        return key == pattern;
    }
    let mut pos = 0;
    for (i, part) in parts.iter().enumerate() {
        if part.is_empty() {
            continue;
        }
        if let Some(p) = key[pos..].find(part) {
            pos += p + part.len();
        } else {
            return false;
        }
    }
    true
}

assert!(simple_glob_match("*", "anything"));
assert!(simple_glob_match("user:*", "user:42"));
assert!(!simple_glob_match("user:*", "config"));
println!("Glob matcher works!");

## 🎯 Key Takeaways

1. **MemoryStorage** implements `Storage` with `HashMap<String, Entry>`
2. GET: lookup + lazy expiration check (return None if expired)
3. SET: insert with optional TTL (expires_at = created_at + ttl)
4. DELETE: remove from HashMap
5. KEYS: pattern matching (prefix with `*`, or full glob)
6. Thread-safe: `Arc<RwLock<HashMap>>` for concurrent access
7. Lazy expiration is simple; active cleanup can be added later

---
**Tomorrow:** Core logic part 2 — persistence and protocol